In [44]:
import os
import rootutils

from tqdm.notebook import tqdm

rootutils.setup_root(os.path.abspath('./'), indicator=".project-root", pythonpath=True, dotenv=True, cwd=True)

# auto-loading of imports from outside scripts
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [45]:
import pandas as pd
import re
import numpy as np

import optuna
from lightning import seed_everything

seed_everything(42, verbose=False)
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [46]:
# df_coord_numbs = pd.read_csv("data_cod/coord_numbs.csv")

df_coord_numbs = pd.read_csv("data_cod/coord_numbs_vdw_3x3x3.csv")

df_coord_numbs.rename(columns={"smiles": "can_smiles"}, inplace=True)

- leaving only organic atoms:

In [47]:
# your allowed elements
organic = {"C", "H", "O", "N", "S", "F", "Cl", "Br", "P"}

# regex to pull out element symbols (Cl, Br or any capital letter + optional lowercase)
_pattern = re.compile(r'Cl|Br|[A-Z][a-z]?')

def smiles_only_organic(smiles: str) -> bool:
    """Return True if every element token in a SMILES string is in our organic set."""
    tokens = _pattern.findall(smiles)
    return all(tok in organic for tok in tokens)

def col_name_ok(col: str) -> bool:
    """
    Return True if, when splitting the column name on dash/–,
    every part is one of our organic elements.
    """
    parts = re.split(r'[–-]', col)
    return all(part in organic for part in parts)

# 1) Filter rows by can_smiles
df_rows = df_coord_numbs[df_coord_numbs['can_smiles'].apply(smiles_only_organic)]

# 2) Build list of columns to keep (and also keep can_smiles itself)
keep_cols = ['id', 'can_smiles'] + [c for c in df_rows.columns if col_name_ok(c)]

# 3) Slice down to just those columns
df_coord_numbs = df_rows[keep_cols]

---
## Merging with temperatures:

In [48]:
# Drop rows and columns that contain only zeros:

rows_only_zeros = df_coord_numbs[(df_coord_numbs == 0).all(axis=1)]
columns_only_zeros = df_coord_numbs.loc[:, (df_coord_numbs == 0).all(axis=0)]

df_coord_numbs = df_coord_numbs.drop(columns=columns_only_zeros.columns, index=rows_only_zeros.index)

In [49]:
df_merged_temp = pd.read_csv("data_cod/cod_bradley_merged.csv")
df_merged_temp = df_merged_temp[df_merged_temp["id"].isin(df_coord_numbs["id"])]

# Only bradley, to use part of it for test:
df_bradley_temp = pd.read_csv("data_cod/bradley_with_cif.csv")
df_bradley_temp = df_bradley_temp[df_bradley_temp["id"].isin(df_coord_numbs["id"])]

---
## Creating Train and Test splits:
- 0.2 of Bradley is test, rest is train

In [50]:
TEST_BRADLEY_FRAC = 0.2

In [51]:
df_test = df_bradley_temp.sample(frac=TEST_BRADLEY_FRAC, random_state=42)

df_train = df_merged_temp.drop(index=df_merged_temp[df_merged_temp["id"].isin(df_test["id"])].index)

In [52]:
df_train = pd.merge(df_train, df_coord_numbs, on=["id", "can_smiles"], how="inner")
df_test = pd.merge(df_test, df_coord_numbs, on=["id", "can_smiles"], how="inner")

**Writing csv with smiles, CNs, Temps, that was extracted to work on temperature**

In [53]:
pd.concat([df_train.drop(columns=["cif_path"]), df_test], ignore_index=True).to_csv("data_cod/coord_numbs_temp_smiles.csv", index=False)

**Organize into data to train on:**

In [54]:
X_train_cn = df_train.drop(columns=["id", "can_smiles", "T", "cif_path"])
X_train_smiles = pd.DataFrame(df_train["can_smiles"])
y_train = df_train["T"].astype(float)

X_test_cn = df_test.drop(columns=["id", "can_smiles", "T"])
X_test_smiles = pd.DataFrame(df_test["can_smiles"])
y_test = df_test["T"].astype(float)

### Turning smiles into features:

In [55]:
from src.utils import create_or_load_data
from src.avail_descriptors import descriptors_all

In [56]:
data_args = {
    "descriptors": descriptors_all,

    "apply_norm": False,
    
    "create_fingerprints": False,
    "temp_column": False,
}

In [57]:
train_label_dummy = [1] * len(df_train)
X_train_smiles["label"] = train_label_dummy

In [58]:
X_train_smiles, _, _ = create_or_load_data(X_train_smiles[:], data_args, "saved_np_obj_with_embed_molecule/cif_data_train", load_data=True)

Loading data from saved_np_obj_with_embed_molecule/cif_data_train


In [59]:
test_label_dummy = [1] * len(df_test)
X_test_smiles["label"] = test_label_dummy

In [61]:
X_test_smiles, _, _ = create_or_load_data(X_test_smiles, data_args, "saved_np_obj_with_embed_molecule/cif_data_test", load_data=True)

Loading data from saved_np_obj_with_embed_molecule/cif_data_test


- Dropping rows from both feature matricies, that have nan in X_smiles to make it fair:

In [62]:
def drop_nan(X, y):
    mask = ~np.isnan(X).any(axis=1)

    X_clean = X[mask]
    y_clean = y[mask]
    
    return X_clean, y_clean

# Dropping rows from X_cn, that have nan in X_smiles to make it fair and more comfortable:
_, X_train_cn = drop_nan(X_train_smiles, X_train_cn)

X_train_smiles, y_train = drop_nan(X_train_smiles, y_train)

In [74]:
assert X_train_cn.shape[0] == X_train_smiles.shape[0] == y_train.shape[0]

---
# Catboost on CN:

In [18]:
import numpy as np

from src.high_level_train_wrap import train_optuna_catboost

In [19]:
best_params = train_optuna_catboost(X_train_cn, y_train, X_test_cn, y_test, n_trials=50, n_jobs=8)

  0%|          | 0/50 [00:00<?, ?it/s]

🏆 New Best Trial 7: R2=0.67349, RMSE=56.00606
🏆 New Best Trial 4: R2=0.69911, RMSE=53.76411
🏆 New Best Trial 1: R2=0.71442, RMSE=52.37817
🏆 New Best Trial 5: R2=0.73624, RMSE=50.33726
🏆 New Best Trial 6: R2=0.74538, RMSE=49.45750
🏆 New Best Trial 12: R2=0.74912, RMSE=49.09282
🏆 New Best Trial 19: R2=0.75792, RMSE=48.22399
🏆 New Best Trial 31: R2=0.76688, RMSE=47.32333


---
# Neural Net on CN:

In [78]:
from argparse import Namespace
import torch

from torch.utils.data import TensorDataset, DataLoader
from src.multitask_nn import RegressionModel

from sklearn.preprocessing import Normalizer

from src.high_level_train_wrap import train_regression_nn

In [79]:
cfg = Namespace(
    project_name="cif_data_coord_numbs",

    batch_size=512,
    lr=1e-3,
    scheduler_gamma=0.99,
    max_epochs=200,

    hid_dim=512,
    depth=5,
    use_residual=True,
    drop=0.2,

    num_workers=0,
    persistent_workers=False,
)

In [80]:
normer_cn = Normalizer()
X_train_cn_norm = normer_cn.fit_transform(X_train_cn.to_numpy())
X_test_cn_norm = normer_cn.transform(X_test_cn.to_numpy())

In [81]:
### with Normalization:
# nn_model_cn = train_regression_nn(
#     X_train=X_train_cn_norm,
#     y_train=y_train.astype(float).to_numpy(),
#     X_val=X_test_cn_norm,
#     y_val=y_test.astype(float).to_numpy(),
#     cfg=cfg,
# )

### without (it is better):
nn_model_cn = train_regression_nn(
    X_train=X_train_cn.to_numpy(),
    y_train=y_train.astype(float).to_numpy(),
    X_val=X_test_cn.to_numpy(),
    y_val=y_test.astype(float).to_numpy(),
    cfg=cfg,
)


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name      | Type              | Params | Mode 
--------------------------------------------------------
0 | shared    | Sequential        | 1.1 M  | train
1 | regressor | Sequential        | 263 K  | train
2 | val_mse   | MeanSquaredError  | 0      | train
3 | val_mae   | MeanAbsoluteError | 0      | train
4 | val_r2    | R2Score           | 0      | train
--------------------------------------------------------
1.4 M     Trainable params
0         Non-trainable params
1.4 M     Total params
5.444     Total estimated model params size (MB)
38        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/aleksandr.varlamov/phase-prediction/.venv/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.
/home/aleksandr.varlamov/phase-prediction/.venv/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.
/home/aleksandr.varlamov/phase-prediction/.venv/lib/python3.10/site-packages/lightning/pytorch/loops/fit_loop.py:310: The number of training batches (11) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=200` reached.


---
# Catboost on Molecular features:

In [96]:
best_params = train_optuna_catboost(X_train_smiles, y_train, X_test_smiles, y_test, n_trials=50, n_jobs=8)

  0%|          | 0/50 [00:00<?, ?it/s]

🏆 New Best Trial 2: R2=0.77713, RMSE=46.27109🏆 New Best Trial 7: R2=0.80717, RMSE=43.03990

🏆 New Best Trial 6: R2=0.82913, RMSE=40.51594
🏆 New Best Trial 9: R2=0.84773, RMSE=38.24678
🏆 New Best Trial 32: R2=0.85412, RMSE=37.43587
🏆 New Best Trial 33: R2=0.85855, RMSE=36.86234


---
# NN on Molecular features:
- First need to normalize features

In [85]:
cfg.project_name="cif_data_molecular"
cfg.lr = 3e-4

cfg.hid_dim = 768
cfg.depth = 7
cfg.use_residual = True
cfg.drop = 0.2

In [86]:
normer = Normalizer()
X_train_smiles_norm = normer.fit_transform(X_train_smiles)
X_test_smiles_norm = normer.transform(X_test_smiles)

In [ ]:
nn_model_smiles = train_regression_nn(
    X_train=X_train_smiles_norm,
    y_train=y_train.to_numpy(dtype=float),
    X_val=X_test_smiles_norm,
    y_val=y_test.to_numpy(dtype=float),
    cfg=cfg,
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name      | Type              | Params | Mode 
--------------------------------------------------------
0 | shared    | Sequential        | 3.6 M  | train
1 | regressor | Sequential        | 591 K  | train
2 | val_mse   | MeanSquaredError  | 0      | train
3 | val_mae   | MeanAbsoluteError | 0      | train
4 | val_r2    | R2Score           | 0      | train
--------------------------------------------------------
4.2 M     Trainable params
0         Non-trainable params
4.2 M     Total params
16.881    Total estimated model params size (MB)
50        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/aleksandr.varlamov/phase-prediction/.venv/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.
/home/aleksandr.varlamov/phase-prediction/.venv/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.
/home/aleksandr.varlamov/phase-prediction/.venv/lib/python3.10/site-packages/lightning/pytorch/loops/fit_loop.py:310: The number of training batches (11) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=200` reached.


---
# Catboost on CN + Molecular features

In [82]:
X_train_concat = np.concatenate([X_train_cn, X_train_smiles], axis=1)

X_test_concat = np.concatenate([X_test_cn, X_test_smiles], axis=1)

In [30]:
# normer = Normalizer()
# X_train_concat_norm = normer.fit_transform(X_train_concat)
# X_test_concat_norm = normer.transform(X_test_concat)

In [83]:
best_params = train_optuna_catboost(X_train_concat, y_train, X_test_concat, y_test, n_trials=100, n_jobs=16)

  0%|          | 0/100 [00:00<?, ?it/s]

🏆 New Best Trial 0: R2=0.77375, RMSE=46.62079
🏆 New Best Trial 13: R2=0.80437, RMSE=43.35139
🏆 New Best Trial 10: R2=0.80509, RMSE=43.27128🏆 New Best Trial 5: R2=0.81163, RMSE=42.53940
🏆 New Best Trial 2: R2=0.81566, RMSE=42.08163

🏆 New Best Trial 1: R2=0.81944, RMSE=41.64890
🏆 New Best Trial 6: R2=0.82088, RMSE=41.48230
🏆 New Best Trial 18: R2=0.82467, RMSE=41.04020
🏆 New Best Trial 9: R2=0.83962, RMSE=39.25249
🏆 New Best Trial 23: R2=0.83976, RMSE=39.23460
🏆 New Best Trial 25: R2=0.84422, RMSE=38.68533
🏆 New Best Trial 26: R2=0.84727, RMSE=38.30467
🏆 New Best Trial 39: R2=0.85835, RMSE=36.88856
🏆 New Best Trial 67: R2=0.85869, RMSE=36.84466
🏆 New Best Trial 87: R2=0.86472, RMSE=36.05030


---
# NN on CN + Molecular Features

In [90]:
# X_train_concat_norm = np.concatenate([X_train_cn_norm, X_train_smiles_norm], axis=1)
# X_test_concat_norm = np.concatenate([X_test_cn_norm, X_test_smiles_norm], axis=1)

In [92]:
normer_concat = Normalizer()
X_train_concat_norm = normer_concat.fit_transform(X_train_concat)
X_test_concat_norm = normer_concat.transform(X_test_concat)

In [94]:
cfg.project_name="cif_data_concat"
cfg.lr = 3e-4

cfg.hid_dim = 256
cfg.depth = 5
cfg.use_residual = True
cfg.drop = 0.2

In [95]:
## with Normaliztion:

nn_model_smiles = train_regression_nn(
    X_train=X_train_concat_norm,
    y_train=y_train.to_numpy(dtype=float),
    X_val=X_test_concat_norm,
    y_val=y_test.to_numpy(dtype=float),
    cfg=cfg,
)

## without Normaliztion:

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name      | Type              | Params | Mode 
--------------------------------------------------------
0 | shared    | Sequential        | 311 K  | train
1 | regressor | Sequential        | 66.0 K | train
2 | val_mse   | MeanSquaredError  | 0      | train
3 | val_mae   | MeanAbsoluteError | 0      | train
4 | val_r2    | R2Score           | 0      | train
--------------------------------------------------------
377 K     Trainable params
0         Non-trainable params
377 K     Total params
1.509     Total estimated model params size (MB)
38        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=200` reached.
